# Notebook 02 - Baseline Systems Before Fine-Tuning

This notebook implements and benchmarks baseline legal analysis systems first.

## What Is This Technique? - Prompt-Only and Few-Shot Baseline Benchmarking

### Definition and Core Concepts
Baseline benchmarking measures what non-fine-tuned systems can do before any adaptation.

### Why Was This Technique Developed?
Without baselines, post-training results are not interpretable.

### What Limitations of Traditional RAG Does It Solve?
RAG can improve factual grounding, but it does not guarantee legal-schema compliance. Baseline prompting reveals this gap explicitly.

### Architecture and Workflow Diagram Explanation
```mermaid
flowchart LR
A[Test legal rows] --> B[Prompt-only]
A --> C[Few-shot]
A --> D[Granite zero-shot]
A --> E[Qwen zero-shot]
B --> F[Predictions]
C --> F
D --> F
E --> F
```

### Component-by-Component Breakdown
1. Prompt templates with schema constraints.
2. Few-shot exemplars.
3. Two base local models.
4. Structured parser and fallback handling.

### When Should It Be Used in Real-World Systems?
Always use before fine-tuning to establish a real lower bound.

### Advantages and Disadvantages
Advantages:
- Low setup cost.
- Fast iteration.
- Useful diagnostic reference.

Disadvantages:
- Output inconsistency.
- Higher schema-break probability.
- Weaker extraction fidelity.

### Comparison Against Standard RAG and Other Implemented Variants
Compared with standard RAG baselines, this benchmark isolates generation behavior without retrieval confounders. It gives a clean before/after fine-tuning comparison.

### Implementation Details and Design Decisions Used in This Project
We run four systems and persist `predictions_*.jsonl` files with latency fields for later analysis.


In [1]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'scripts').exists() and (p / 'configs').exists()), cwd)
print('Project root:', ROOT)

def run_py(script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / script), *args]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))


Project root: /home/ahmad/AI/Legal-Document-Summarizer-Clause-Analyzer


In [2]:
run_dir = ROOT / 'artifacts/runs/balanced_local_v1'
if not list(run_dir.glob('predictions_*.jsonl')):
    run_py('scripts/run_baselines.py', '--config', 'configs/default.yaml')
sorted([p.name for p in run_dir.glob('predictions_*.jsonl')])

['predictions_few_shot.jsonl',
 'predictions_fine_tuned_adapter.jsonl',
 'predictions_granite_zero_shot.jsonl',
 'predictions_prompt_only.jsonl',
 'predictions_qwen_zero_shot.jsonl']

In [3]:
samples = []
for path in sorted(run_dir.glob('predictions_*.jsonl')):
    with path.open() as f:
        first = json.loads(next(f))
        samples.append({
            'system': path.stem.replace('predictions_', ''),
            'summary': first['prediction']['executive_summary'][:180],
            'risk_level': first['prediction']['risk_level'],
            'latency_seconds': first.get('latency_seconds', 0.0),
        })
pd.DataFrame(samples)

,system,summary,risk_level,latency_seconds
0,few_shot,No summary generated.,medium,2.442618
1,fine_tuned_adapter,No summary generated.,medium,8.221680
2,granite_zero_shot,No summary generated.,medium,4.044479
3,prompt_only,No summary generated.,medium,7.246250
4,qwen_zero_shot,No summary generated.,medium,2.469156


## Baseline Observations

Use these baseline outputs as the direct comparator when judging fine-tuned improvements in quality, schema consistency, and latency.